# 00 — Placebo sample from IFLS3 (2000) → IFLS4 (2007) → IFLS5 (2014)

Builds `umkm_ifls3_placebo.csv`, used by notebook `01` (Section D, closure placebo; Additional file 1, Table A5) and notebook `02` (X3 pre-trend placebo, X4 subgroups).

* Sample: households with a non-farm enterprise in 2000, traced to 2007 (via `pidlink`) and to 2014 (via `htrack`).
* Placebo outcome: enterprise closure between 2000 and 2007 (same definition as the main outcome).
* Treatment: non-family credit in 2014 (`bh07`, same as the main analysis).
* Covariates: 2000 characteristics only.

Inputs: IFLS3 (`hh00_b2`: `b2_nt1`, `b2_nt2`; `hh00_b3b`: `b3b_bh1`, `b3b_bh2`; `hh00_bk`: `bk_ar1`, `bk_sc`), IFLS4 (`hh07_bk`: `bk_ar1`; `hh07_b2`: `b2_nt1`, `b2_nt2`), IFLS5 (`hh14_trk`: `htrack`; `hh14_b2`: `b2_bh`). Set `IFLS_ROOT` / `IFLS_OUT` as environment variables outside Kaggle.

Note: `pendidikan_kepala00` is stored as the IFLS education-level code (`ar16`); notebooks `01` and `02` convert it to years of schooling. Code comments are in Indonesian.


In [ ]:
# =====================================================================
# PERSIAPAN DATA IFLS3 (2000) -> IFLS4 (2007) untuk UJI PLACEBO / PRA-TREN
# Pasangan dari prep_ifls.py (IFLS4 -> IFLS5). Definisi dibuat sama:
#   - unit        = rumah tangga (RT) dengan usaha non-pertanian
#   - usaha tutup = RT tidak lagi punya usaha yang masih beroperasi (nt01 != 1 atau tidak ada nt18 == 1)
#   - kredit      = pinjaman non-keluarga 12 bln terakhir, bh07 di IFLS5 (SAMA dengan treatment utama)
# Desain placebo:
#   sampel     = RT dengan usaha non-pertanian di 2000, terlacak ke 2007 dan 2014
#   outcome    = usaha_tutup_0007  (tutup antara 2000 dan 2007)
#   treatment  = kredit 2014 (masa depan relatif terhadap outcome) -> efek sebenarnya harus 0
#   kovariat   = kondisi 2000 saja (pra-outcome). JANGAN pakai kovariat 2007.
# Output: /kaggle/working/umkm_ifls3_placebo.csv  (+ panel_2000.csv sebagai checkpoint)
# =====================================================================
import os, glob, warnings
import numpy as np, pandas as pd
warnings.filterwarnings("ignore")

ROOT = os.environ.get("IFLS_ROOT", "/kaggle/input")
OUT_DIR = os.environ.get("IFLS_OUT", "/kaggle/working")
STAGE1_ONLY = os.environ.get("STAGE1_ONLY", "0") == "1"   # hanya bangun panel 2000 (tanpa 2007/2014)

def find(folder_hint, fname):
    hits = [f for f in glob.glob(f"{ROOT}/**/{fname}.dta", recursive=True) if folder_hint in f]
    if not hits:
        raise FileNotFoundError(f"{fname}.dta di folder '{folder_hint}' tidak ditemukan")
    return sorted(hits, key=len)[0]

def rd(folder_hint, fname, cols=None):
    df = pd.read_stata(find(folder_hint, fname), convert_categoricals=False)
    df.columns = [c.lower() for c in df.columns]
    if cols:
        df = df[[c for c in cols if c in df.columns]]
    return df

def sid(s):
    return s.astype(str).str.strip().str.replace(r"\.0$", "", regex=True)

def num(s):
    return pd.to_numeric(s, errors="coerce")

ihs = np.arcsinh

In [ ]:
# ---------------------------------------------------------------------
# 1. PANEL 2000: RT dengan usaha non-pertanian
# ---------------------------------------------------------------------
nt1 = rd("hh00_b2", "b2_nt1", ["hhid00", "nt01"])
nt1["hhid00"] = sid(nt1.hhid00)
base = nt1.loc[num(nt1.nt01) == 1, ["hhid00"]].drop_duplicates()
print("RT dengan usaha non-pertanian 2000:", len(base))

nt2 = rd("hh00_b2", "b2_nt2")
nt2["hhid00"] = sid(nt2.hhid00)
for c in ["nt07", "nt09", "nt22", "nt23", "nt10a", "nt10b", "nt10c1", "nt10c4", "nt10h", "nt15yr", "nt05b"]:
    nt2[c] = num(nt2[c]) if c in nt2 else np.nan
op = nt2[num(nt2.nt18).fillna(1) == 1].copy()                       # masih beroperasi, sama dengan 2007
op["aset"] = op[["nt10a", "nt10b", "nt10c1", "nt10c4", "nt10h"]].sum(axis=1, min_count=1)
op["pekerja"] = op[["nt22", "nt23"]].sum(axis=1, min_count=1)
op["umur"] = 2000 - op["nt15yr"].where(op["nt15yr"].between(1900, 2000))
g00 = op.groupby("hhid00").agg(
    omzet00=("nt07", "sum"), laba00=("nt09", "sum"), pekerja00=("pekerja", "sum"),
    aset00=("aset", "sum"), umur_usaha00=("umur", "max"), n_usaha00=("nt07", "size"),
    usaha_di_rumah00=("nt05b", lambda s: float((s == 1).any())),
).reset_index()
base = base.merge(g00, on="hhid00", how="left")

# pinjaman 2000 (individu -> RT). bh02: pinjam dari non-keluarga/teman 12 bln; bh05: dapat pinjaman 12 bln
bh1 = rd("hh00_b3b", "b3b_bh1", ["hhid00", "bh02", "bh05"])
bh1["hhid00"] = sid(bh1.hhid00)
a = bh1.groupby("hhid00").agg(pinjam_nonkel00=("bh02", lambda s: float((num(s) == 1).any())),
                              pinjam00=("bh05", lambda s: float((num(s) == 1).any()))).reset_index()
bh2 = rd("hh00_b3b", "b3b_bh2", ["hhid00", "bh14"])
bh2["hhid00"] = sid(bh2.hhid00)
amt = bh2.assign(bh14=num(bh2.bh14)).groupby("hhid00").bh14.sum().rename("jumlah_pinjam00").reset_index()
base = base.merge(a, on="hhid00", how="left").merge(amt, on="hhid00", how="left")
base["jumlah_pinjam00"] = base.jumlah_pinjam00.fillna(0)
base["ihs_pinjam00"] = ihs(base.jumlah_pinjam00)

# lokasi 2000 (kode kabupaten 2000, perlu harmonisasi dengan kode 2007/2014)
sc = rd("hh00_bk", "bk_sc", ["hhid00", "sc01", "sc02", "sc05"])
sc["hhid00"] = sid(sc.hhid00)
sc["perkotaan00"] = (num(sc.sc05) == 1).astype(float)
sc = sc.rename(columns={"sc01": "prov00", "sc02": "kab00"})
base = base.merge(sc[["hhid00", "prov00", "kab00", "perkotaan00"]].drop_duplicates("hhid00"), on="hhid00", how="left")

# kepala RT 2000 dan ukuran RT dari roster. Kode ar02b/ar07/ar09/ar16 dipakai sama dengan prep_ifls.py:
# CEK di hh00_cbk (BK) bahwa kodenya sama sebelum dipercaya.
ar = rd("hh00_bk", "bk_ar1", ["hhid00", "pidlink", "ar02b", "ar07", "ar09", "ar16"])
ar["hhid00"] = sid(ar.hhid00)
hhs = ar.groupby("hhid00").size().rename("hhsize00").reset_index()
head = ar[num(ar.ar02b) == 1].drop_duplicates("hhid00")
head = head.assign(kepala_perempuan00=(num(head.ar07) == 3).astype(float),
                   usia_kepala00=num(head.ar09).where(num(head.ar09).between(10, 100)),
                   pendidikan_kepala00=num(head.ar16).where(num(head.ar16) < 90))
base = (base.merge(hhs, on="hhid00", how="left")
            .merge(head[["hhid00", "kepala_perempuan00", "usia_kepala00", "pendidikan_kepala00"]], on="hhid00", how="left"))

# konteks desa 2000: jumlah lembaga kredit (CF00). Sebelum dipakai, cek kuesioner CF00 untuk
# variabel setara bk1_g (gtype/g3a/g3b) dan kunci commid00; di sini DILEWATI.

base.to_csv(f"{OUT_DIR}/panel_2000.csv", index=False)
print(f"Checkpoint panel 2000: {len(base)} RT ->", f"{OUT_DIR}/panel_2000.csv")
if STAGE1_ONLY:
    raise SystemExit(0)

In [ ]:
# ---------------------------------------------------------------------
# 2. LACAK 2000 -> 2007 lewat pidlink (tidak ada tabel hhid00->hhid07 di berkas yang diunduh)
# ---------------------------------------------------------------------
r00 = ar[["hhid00", "pidlink"]].dropna()
r07 = rd("hh07_bk", "bk_ar1", ["hhid07", "pidlink"])
r07["hhid07"] = sid(r07.hhid07)
r00["pidlink"] = sid(r00.pidlink); r07["pidlink"] = sid(r07.pidlink)
m = r00.merge(r07, on="pidlink").groupby(["hhid00", "hhid07"]).size().rename("n_shared").reset_index()
n_split = m.groupby("hhid00").hhid07.nunique().rename("n_hhid07").reset_index()
m = m.sort_values(["hhid00", "n_shared"], ascending=[True, False]).drop_duplicates("hhid00")   # RT asal = paling banyak anggota sama
base = base.merge(m[["hhid00", "hhid07"]], on="hhid00", how="inner").merge(n_split, on="hhid00", how="left")
base = base.drop_duplicates("hhid07")                                                           # satu RT 2000 per RT 2007
print("Terlacak ke 2007:", len(base))

In [ ]:
# ---------------------------------------------------------------------
# 3. OUTCOME PLACEBO: usaha tutup 2000 -> 2007 (definisi sama dengan prep_ifls.py)
# ---------------------------------------------------------------------
nt1_07 = rd("hh07_b2", "b2_nt1", ["hhid07", "nt01"]); nt1_07["hhid07"] = sid(nt1_07.hhid07)
nt2_07 = rd("hh07_b2", "b2_nt2"); nt2_07["hhid07"] = sid(nt2_07.hhid07)
nt2_07["nt18"] = num(nt2_07.nt18) if "nt18" in nt2_07 else np.nan
n07 = nt2_07[nt2_07.nt18.fillna(1) == 1].groupby("hhid07").size().rename("n_usaha07_op").reset_index()
base = base.merge(nt1_07, on="hhid07", how="left").merge(n07, on="hhid07", how="left")
base["usaha_tutup_0007"] = ((num(base.nt01) != 1) | base.n_usaha07_op.isna()).astype(int)

In [ ]:
# ---------------------------------------------------------------------
# 4. TREATMENT 2014 (sama dengan prep_ifls.py), lewat htrack IFLS5
# ---------------------------------------------------------------------
ht = rd("hh14_trk", "htrack", ["hhid07", "hhid14", "result14", "hwt14la", "hwt14l_", "splitoff14"])
ht["hhid07"] = sid(ht.hhid07); ht["hhid14"] = sid(ht.hhid14)
ht = ht[(ht.hhid14 != "") & (ht.hhid14 != "nan")]
ht["_ord"] = num(ht.splitoff14).fillna(0) if "splitoff14" in ht else 0
ht = ht.sort_values(["hhid07", "_ord", "hhid14"]).drop_duplicates("hhid07")
base = base.merge(ht, on="hhid07", how="inner")
print("Terlacak ke 2014:", len(base))

bh14 = rd("hh14_b2", "b2_bh", ["hhid14", "bh02", "bh07", "bh10"]); bh14["hhid14"] = sid(bh14.hhid14)
bh14["kredit"] = (num(bh14.bh07) == 1).astype(int)
bh14["mencoba_pinjam"] = (num(bh14.bh02) == 1).astype(int)
base = base.merge(bh14[["hhid14", "kredit", "mencoba_pinjam"]], on="hhid14", how="inner")

In [ ]:
# ---------------------------------------------------------------------
# 5. FINAL
# ---------------------------------------------------------------------
base["weight"] = num(base.hwt14la).fillna(num(base.get("hwt14l_")))
base["weight"] = base["weight"].fillna(base["weight"].median())
for c, f in [("omzet00", ihs), ("laba00", ihs), ("aset00", ihs)]:
    base["ihs_" + c] = f(base[c].fillna(0))
base["id"] = base.hhid07

covs = ["ihs_omzet00", "ihs_laba00", "pekerja00", "ihs_aset00", "umur_usaha00", "n_usaha00", "usaha_di_rumah00",
        "pinjam_nonkel00", "pinjam00", "ihs_pinjam00", "hhsize00", "kepala_perempuan00", "usia_kepala00",
        "pendidikan_kepala00", "perkotaan00"]
keep = ["id", "hhid00", "hhid07", "hhid14", "prov00", "kab00", "weight", "kredit", "mencoba_pinjam",
        "usaha_tutup_0007", "n_hhid07"] + covs
df = base[[c for c in keep if c in base.columns]].copy()
for c in [c for c in covs if c in df.columns]:
    if df[c].isna().any():
        df[f"{c}_miss"] = df[c].isna().astype(int); df[c] = df[c].fillna(df[c].median())
df.to_csv(f"{OUT_DIR}/umkm_ifls3_placebo.csv", index=False)

print("\n=== RINGKASAN PLACEBO ===")
print(f"N = {len(df):,} | kabupaten 2000 = {df.kab00.nunique()} | provinsi = {df.prov00.nunique()}")
print(f"Treatment (kredit 2014) = {df.kredit.mean():.1%} | usaha tutup 2000-2007 = {df.usaha_tutup_0007.mean():.1%}")
print(df.groupby("kredit").usaha_tutup_0007.agg(["mean", "size"]).round(3))
print("\nMerge dengan umkm_ifls.csv lewat hhid07 hanya bila ingin kontrol kabupaten/koordinat 2014 yang sama.")